# Milestone 11 — Integrated reversible D2Q25
Run from the experiment root or milestone11 directory. This notebook is a runnable guide; reported results come from run.py. Basis gate simulation is not QPU execution.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
root = Path.cwd()
if root.name == "milestone11": root = root.parent
sys.path.insert(0, str(root))
from milestone11.collision import CollisionOracle
from milestone11.solver import Solver
from milestone11 import reference as ref


## Local collision
Input populations are nonnegative integer words. The arithmetic circuit conserves their sum and both components of momentum and uncomputes its scratch registers.

In [ ]:
o = CollisionOracle(32)
f = np.zeros((2,25), dtype=int)
f[0,12] = 48
f[1,13] = 20
z, flags = o.evaluate(f)
assert np.array_equal(z.sum(-1), f.sum(-1))
assert np.array_equal(z @ ref.V, f @ ref.V)
print(z, flags, o.nqubits, len(o.gates))

## Small integrated simulation
A closed 4×8 grid with one obstacle. Physical evolution uses the gate-based collision and exact streaming permutation.

In [ ]:
mask = np.zeros((4,8), bool)
mask[1,3] = True
h = np.zeros(mask.shape); h[:,:2] = 1.5
f = ref.initialize(h, 32, o.w)
c = Solver(mask,32,oracle=o)
for _ in range(5): f, flags = c.step(f)
print(ref.moments(f/32)[0])

## Full benchmark results
The analytical comparison is independent of the fixed-point implementation. The nonzero errors and unrealizable pressure states are important limitations.

In [ ]:
report = json.loads((root / "milestone11/results/report.json").read_text())
for row in report["strip"]:
    print(row["scale"], row["final"]["ritter_rmse"], row["final"]["float_ritter_rmse"])
from IPython.display import display, Image
display(Image(filename=str(root / "milestone11/results/ritter_comparison.png")))